In [27]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import pandas as pd


# 1. Load Data (SMS Spam Collection)
twitter = "/Users/suraj/Downloads/twitter_training.csv";
df = pd.read_csv(twitter)

In [28]:
df

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [29]:
df.columns = ['ID', 'Entity', 'Sentiment', 'Tweet_Content']

In [30]:
df

,ID,Entity,Sentiment,Tweet_Content
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [31]:
df["Sentiment"].value_counts()

Sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ID             74681 non-null  int64
 1   Entity         74681 non-null  str  
 2   Sentiment      74681 non-null  str  
 3   Tweet_Content  73995 non-null  str  
dtypes: int64(1), str(3)
memory usage: 11.4 MB


In [33]:
df.isnull().sum()

ID                 0
Entity             0
Sentiment          0
Tweet_Content    686
dtype: int64

In [34]:
df.duplicated().sum()

np.int64(2700)

In [35]:
df = df.drop_duplicates()

In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:
df = df.dropna(subset=['Tweet_Content'])

In [38]:
df.isnull().sum()

ID               0
Entity           0
Sentiment        0
Tweet_Content    0
dtype: int64

In [39]:
df.info()

<class 'pandas.DataFrame'>
Index: 71655 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ID             71655 non-null  int64
 1   Entity         71655 non-null  str  
 2   Sentiment      71655 non-null  str  
 3   Tweet_Content  71655 non-null  str  
dtypes: int64(1), str(3)
memory usage: 11.7 MB


In [40]:
display(df)

,ID,Entity,Sentiment,Tweet_Content
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [41]:
df.columns

Index(['ID', 'Entity', 'Sentiment', 'Tweet_Content'], dtype='str')

In [42]:
categorical_cols = ['Entity', 'Sentiment', 'Tweet_Content']

In [43]:
label_encoders = {}

In [44]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"Encoded {col}: {df[col].nunique()} unique values")

Encoded Entity: 32 unique values
Encoded Sentiment: 4 unique values
Encoded Tweet_Content: 69490 unique values


In [45]:
display(df)

,ID,Entity,Sentiment,Tweet_Content,Entity_encoded,Sentiment_encoded,Tweet_Content_encoded
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,4,3,27233
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,4,3,64618
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,4,3,64602
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,4,3,64617
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,4,3,64616
...,...,...,...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...,21,3,36808
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...,21,3,36807
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...,21,3,36810
74679,9200,Nvidia,Positive,Just realized between the windows partition of...,21,3,36803


In [46]:
df.columns

Index(['ID', 'Entity', 'Sentiment', 'Tweet_Content', 'Entity_encoded',
       'Sentiment_encoded', 'Tweet_Content_encoded'],
      dtype='str')

In [47]:
correlation_matrix = df[['ID','Entity_encoded', 'Sentiment_encoded']].corr()
print(correlation_matrix)

                         ID  Entity_encoded  Sentiment_encoded
ID                 1.000000        0.940583           0.014623
Entity_encoded     0.940583        1.000000           0.012606
Sentiment_encoded  0.014623        0.012606           1.000000


In [48]:
df["Sentiment"].value_counts()

Sentiment
Negative      21698
Positive      19712
Neutral       17708
Irrelevant    12537
Name: count, dtype: int64

In [49]:
df["Sentiment_encoded"].value_counts()

Sentiment_encoded
1    21698
3    19712
2    17708
0    12537
Name: count, dtype: int64

In [50]:
rating_map = {
    0: 0, 1: 0, 2: 0,   # 0-2 become 0 (Negative)
    3: 1       # 3 become 1 (Positive)
}

df['Sentiment_encoded_class'] = df['Sentiment_encoded'].map(rating_map)

In [53]:
X = df['Entity']
y = df['Sentiment_encoded_class']

In [54]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vect = CountVectorizer()
Model_train = vect.fit_transform(X_train)
Model_test = vect.transform(X_test)      

nb = ComplementNB()
nb.fit(Model_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueOnly used in edge case with a single class in the training set.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. Not used.",None
,"norm norm: bool, default=FalseWhether or not a second normalization of the weights is performed. Thedefault behavior mirrors the implementations found in Mahout and Weka,which do not follow the full algorithm described in Table 9 of thepaper.",False


In [55]:
y_pred = nb.predict(Model_test)

In [56]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[6851 3569]
 [1768 2143]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.66      0.72     10420
           1       0.38      0.55      0.45      3911

    accuracy                           0.63     14331
   macro avg       0.59      0.60      0.58     14331
weighted avg       0.68      0.63      0.64     14331



In [ ]:
df

,ID,Entity,Sentiment,Tweet_Content,Entity_encoded,Sentiment_encoded,Tweet_Content_encoded,Sentiment_encoded_class
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,4,3,27233,1
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,4,3,64618,1
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,4,3,64602,1
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,4,3,64617,1
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,4,3,64616,1
...,...,...,...,...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...,21,3,36808,1
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...,21,3,36807,1
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...,21,3,36810,1
74679,9200,Nvidia,Positive,Just realized between the windows partition of...,21,3,36803,1


In [ ]:
total_rows = len(df["Sentiment_encoded_class"])

counts = df["Sentiment_encoded_class"].value_counts()

print(f"Total rows: {total_rows}")
print("--- Counts per category ---")
print(counts)

Total rows: 71655
--- Counts per category ---
Sentiment_encoded_class
0    51943
1    19712
Name: count, dtype: int64


In [59]:
df["Entity"].value_counts()

Entity
TomClancysRainbowSix                 2328
Verizon                              2319
MaddenNFL                            2315
CallOfDuty                           2314
Microsoft                            2304
WorldOfCraft                         2300
NBA2K                                2299
LeagueOfLegends                      2296
TomClancysGhostRecon                 2291
Facebook                             2289
ApexLegends                          2278
johnson&johnson                      2257
Battlefield                          2255
Amazon                               2249
CallOfDutyBlackopsColdWar            2242
FIFA                                 2238
Dota2                                2225
Overwatch                            2220
Hearthstone                          2219
HomeDepot                            2216
GrandTheftAuto(GTA)                  2208
Borderlands                          2205
Xbox(Xseries)                        2201
Google                     

In [62]:
custom_message = ["AssassinsCreed"] 

custom_vec = vect.transform(custom_message)
predictions = nb.predict(custom_vec)  

class_map = {0: 'Negative', 1: 'Positive'}

for i, j in zip(custom_message, predictions):
    label = class_map[j]
    print(f"The model predicts Entity '{i}' is: {label}")

The model predicts Entity 'AssassinsCreed' is: Positive


In [ ]:
print(predictions)

[1]


In [ ]:
df["Tweet_Content"][0]

'I am coming to the borders and I will kill you all,'

In [63]:
entity_data = df[df["Entity"] == "Dota2"]
print(entity_data)

         ID Entity Sentiment  \
13997  2801  Dota2   Neutral   
13998  2801  Dota2   Neutral   
13999  2801  Dota2   Neutral   
14000  2801  Dota2   Neutral   
14001  2801  Dota2   Neutral   
...     ...    ...       ...   
16356  3200  Dota2  Positive   
16357  3200  Dota2  Positive   
16358  3200  Dota2  Positive   
16359  3200  Dota2  Positive   
16360  3200  Dota2  Positive   

                                           Tweet_Content  Entity_encoded  \
13997  With so many awesome personalities, the  . est...               9   
13998  With so many great personalities. estnn.com / ...               9   
13999  With so many stunning personalities like est.e...               9   
14000  With so these standout personalities, the . es...               9   
14001  With so so many awesome personalities, thank t...               9   
...                                                  ...             ...   
16356  Finally!!!! Fixer Schedule!!! God Bless GabeN ...               9   
16357  